In [61]:
import torch
from torch import nn
import numpy as np

1. Metodo per creare il bit pattern: viene convertito la variabile num in un numero binario tramite la funzione bin() e tramite zfill(9) possiamo riempire di zeri il numero binario per avere 9 bit. Dopodichè viene creato un array numpy composto dai bit del numero binario, convertiti in float e creato il tensore

In [62]:
def create_kernel(num):
  binary_num = bin(num)[2:].zfill(9)
  np_array = np.array([float(bit) for bit in binary_num], dtype=np.float32)
  kernel = torch.from_numpy(np_array.reshape(3, 3))
  kernel[kernel==0.0]=-1.0
  return kernel

kernel = create_kernel(3)
print(kernel)

tensor([[-1., -1., -1.],
        [-1., -1., -1.],
        [-1.,  1.,  1.]])


2. Metodo per ottenere la convoluzione: vengono creati tutti i bit pattern possibili (kernels) e aggiunta una dimensione (tramite unsqueeze(1)) per essere dato in input alla rete convolutiva. Il metodo restituisce la rete e i bit pattern

In [63]:
def create_conv_detector():
  kernels = [create_kernel(i) for i in range(512)]
  tensor_kernel = torch.stack(kernels).unsqueeze(1)
  net = nn.Conv2d(1,512,3,padding='valid',bias=False)
  with torch.no_grad():
    net.weight.copy_(tensor_kernel)
  return net,tensor_kernel

3. Classe per il FeatureDetector: nel costruttore istanzio i kernel e la rete convolutiva, inoltre tengo il conteggio degli uni (target_scores) contando sulle righe che sulle colonne (dim=(2,3)). Infine al buffer del modello (register_buffer) aggiungo il parametro target_score perché così non viene aggiornato dagli ottimizzatori e non viene coinvolto quindi nel calcolo del gradiente. *target_scores.view(1,512,1,1)* trasforma il tensore di target_scores di forma (512,1) a un tensore di forma (1,512,1,1) dove:

*   1: dimensione del batch
*   512: numero dei canali/feature
*   1: altezza delle feauture map
*   1: larghezza delle feature map

Ciò viene fatto per rendere target_scores compatibile con l'output della rete convolutiva (out) tramite meccanismo di broadcasting all'interno della sintassi *detection=(out==self.target_scores).float()* per poter confrontare out con target_scores, estendendo quest'ultimo le sue dimensioni spaziali (altezza e larghezza)


In [64]:
class FeatureDetector(torch.nn.Module):
  def __init__(self):
    super().__init__()
    self.conv, kernels = create_conv_detector()
    target_scores = (kernels == 1.0).sum(dim=(2, 3)).float()
    self.register_buffer('target_scores', target_scores.view(1, 512, 1, 1))

  def forward(self,batch):
    out=self.conv(batch)
    detection=(out==self.target_scores).float()
    return detection

Testing del feature detector con un immagine di soli zeri

In [65]:
batch_size=1
weight_image=8
height_image=8
batch_data=torch.zeros(weight_image,height_image)
batch=torch.reshape(batch_data,(batch_size,1,height_image,weight_image))
featureDet=FeatureDetector()
detect=featureDet(batch)
print(detect)

tensor([[[[1., 1., 1., 1., 1., 1.],
          [1., 1., 1., 1., 1., 1.],
          [1., 1., 1., 1., 1., 1.],
          [1., 1., 1., 1., 1., 1.],
          [1., 1., 1., 1., 1., 1.],
          [1., 1., 1., 1., 1., 1.]],

         [[0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0.]],

         [[0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0.]],

         ...,

         [[0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0.]],

         [[0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0.],
     

Testing del feature detector con un immagine di soli uni

In [66]:
batch_data=torch.ones(weight_image,height_image)
batch=torch.reshape(batch_data,(batch_size,1,height_image,weight_image))
detect=featureDet(batch)
print(detect)

tensor([[[[0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0.]],

         [[0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0.]],

         [[0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0.]],

         ...,

         [[0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0.]],

         [[0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0.],
     

Feature Detector con un batch di 4 immagini generate random

In [67]:
batch_size=4
weight_image=8
height_image=8
batch=torch.randint(0,2,(batch_size,1,height_image,weight_image)).float()
featureDet=FeatureDetector()
detect=featureDet(batch)

Qui vado a stampare tutte le immagini, le feature map in cui viene rilevato almeno un pattern e il bit pattern rilevato

In [68]:
print("Matrici rilevate con almeno un 1.0 e i loro bit pattern associati:\n")
#Vengono stampate le immagini del batch
for batch_idx in range(batch_size):
  print(f"Analisi immagine {batch_idx+1}")
  print(batch[batch_idx].squeeze())
  print("\n")
  #Per ogni immagine vengono stampati le feature map in cui vengono rilevati determinati bit pattern
  for i in range(512):
    if (detect[batch_idx,i,:,:]==1.0).any():
      print(f"Rilevamento per il bit pattern {i}")
      print("Feature Map Rilevata")
      print(detect[batch_idx,i,:,:])
  #Viene stampato il bit pattern rilevato
      kernel_pattern=featureDet.conv.weight[i].squeeze().clone()
      kernel_pattern[kernel_pattern==-1.0]=0.0
      print("Bit Pattern Associato (kernel 3x3)")
      print(kernel_pattern)
      print("\n")

Matrici rilevate con almeno un 1.0 e i loro bit pattern associati:

Analisi immagine 1
tensor([[0., 1., 1., 0., 0., 1., 0., 1.],
        [0., 1., 0., 1., 0., 0., 1., 1.],
        [1., 0., 0., 0., 1., 0., 1., 1.],
        [1., 0., 0., 0., 0., 1., 0., 0.],
        [1., 1., 1., 1., 0., 0., 1., 0.],
        [0., 0., 1., 0., 0., 0., 1., 0.],
        [1., 1., 1., 0., 1., 0., 1., 1.],
        [0., 1., 0., 0., 0., 0., 0., 0.]])


Rilevamento per il bit pattern 7
Feature Map Rilevata
tensor([[0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.]])
Bit Pattern Associato (kernel 3x3)
tensor([[0., 0., 0.],
        [0., 0., 0.],
        [1., 1., 1.]], grad_fn=<IndexPutBackward0>)


Rilevamento per il bit pattern 16
Feature Map Rilevata
tensor([[0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 

# TODO

1. Scrivere una funzione che prende in input un numero naturale fra 0 e 511 e restituisce il tensore 3x3 che corrisponde al bit pattern.
2. Scrivere una funzione (che usa la prima funzione) che genera tutte le matrici che corrispondono ai bit pattern da 0 a 511. Restituisce una convoluzione PyTorch che effetua il rilevamento.
3. Scrivere una classe (che eretita da torch.nn.Module) che usa le prime due funzioni e implementa il feature detector. Il metodo forward() di questa classe prenderà in input un batch di immagini (Bx1xHxW) e restituisce un tensore di dimensione (Bx512x(H-2)x(W-2)) dove in ogni canale in output c'è un 1.0 dove il feature che corrisponde a quella dimensione è stato rilevato.